# RL for Optimal Trade Execution

A PPO agent that makes per-second execution decisions based on the current order book state.
Instead of predicting 60 seconds ahead, the agent reacts to what it sees right now.

In [1]:
import os, sys, subprocess

# Colab setup
if os.path.exists("/content"):
    # Always ensure we're in the right directory
    repo_dir = "/content/Ultramarin"

    # Clone repo if needed (for simulate_walk_the_book.py)
    if not os.path.isdir(os.path.join(repo_dir, "utils")):
        subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                        "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                       cwd="/content")

    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)

    # Mount Drive and copy data if needed
    dst = os.path.join(repo_dir, "data")
    if not os.path.isdir(dst) or not any(
        d for d in os.listdir(dst) if os.path.isdir(os.path.join(dst, d))
    ):
        from google.colab import drive
        drive.mount('/content/drive')
        src = "/content/drive/MyDrive/data"
        os.makedirs(dst, exist_ok=True)
        subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
        print("Data copied.")
    else:
        print(f"Data already present at {dst}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gymnasium", "stable-baselines3"], capture_output=True)
print("Setup complete.")

Data already present at /content/Ultramarin/data
Setup complete.


In [2]:
import numpy as np
import pandas as pd
import math
import warnings
import random
from pathlib import Path

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

from data.simulate_walk_the_book import simulate_walk_the_book

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# === Change this to switch currencies ===
data_asset = "DOGEUSDT"

KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}

volume_to_fill = KNOWN_VOLUMES[data_asset]
K_SECONDS = OPTIMAL_K[data_asset]

root = Path("data")
Y_path = root / data_asset / "y_train.parquet"

print(f"Currency: {data_asset}")
print(f"Volume to fill: {volume_to_fill}")
print(f"TWAP-K baseline: K={K_SECONDS}")

Currency: DOGEUSDT
Volume to fill: 60349.0
TWAP-K baseline: K=20


## Data Loading

Load Y data (the 60-second execution window), split into train/val using the same
chrono split as mhf-final (sort IDs, 80/20 split), and pre-extract LOB arrays per instrument.

In [ ]:
ASK_PRICE_COLS = [f'ask_price_{i}' for i in range(1, 6)]
ASK_VOL_COLS = [f'ask_vol_{i}' for i in range(1, 6)]
BID_PRICE_COLS = [f'bid_price_{i}' for i in range(1, 6)]
BID_VOL_COLS = [f'bid_vol_{i}' for i in range(1, 6)]

Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])

# Get valid instrument IDs (exactly 60 rows)
id_counts = Y_raw.groupby("anonymized_id").size()
valid_ids = sorted(id_counts[id_counts == 60].index.tolist())

# Chrono split: same logic as train_val() — sort IDs, take last 20% as val
split_idx = int(len(valid_ids) * 0.8)
train_ids = valid_ids[:split_idx]
val_ids = valid_ids[split_idx:]

print(f"Valid instruments: {len(valid_ids)}")
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}")


def extract_instruments(ids, df):
    """Pre-extract LOB arrays for each instrument. Forward-fills NaN prices (like mhf-final)."""
    instruments = []
    skipped = 0
    for aid in ids:
        df_inst = df[df["anonymized_id"] == aid].sort_values("time_in_hour")
        if len(df_inst) != 60:
            continue

        ask_prices = df_inst[ASK_PRICE_COLS].to_numpy(dtype=np.float64)
        bid_prices = df_inst[BID_PRICE_COLS].to_numpy(dtype=np.float64)
        ask_vols = df_inst[ASK_VOL_COLS].to_numpy(dtype=np.float64)
        bid_vols = df_inst[BID_VOL_COLS].to_numpy(dtype=np.float64)

        # Fill NaN volumes with 0 (empty level)
        ask_vols = np.nan_to_num(ask_vols, nan=0.0)
        bid_vols = np.nan_to_num(bid_vols, nan=0.0)

        # Forward-fill NaN prices at ALL levels (including level 1)
        # Same approach as mhf-final's pipeline — don't discard instruments
        for arr in [ask_prices, bid_prices]:
            for col in range(5):
                for row in range(60):
                    if np.isnan(arr[row, col]):
                        if row > 0 and not np.isnan(arr[row - 1, col]):
                            arr[row, col] = arr[row - 1, col]
                        elif col > 0 and not np.isnan(arr[row, col - 1]):
                            # Fall back to previous level at same timestep
                            arr[row, col] = arr[row, col - 1]

        # Skip only if level-1 prices are STILL NaN after forward-fill
        # (i.e., the very first second has NaN and there's nothing to fill from)
        if np.isnan(ask_prices[:, 0]).any() or np.isnan(bid_prices[:, 0]).any():
            skipped += 1
            continue

        # Fill any remaining NaN deeper-level prices with level-1
        for col in range(1, 5):
            for arr in [ask_prices, bid_prices]:
                mask = np.isnan(arr[:, col])
                if mask.any():
                    arr[mask, col] = arr[mask, 0]

        close_vals = df_inst["close"].dropna()
        if len(close_vals) == 0:
            skipped += 1
            continue

        instruments.append({
            "id": aid,
            "ask_prices": ask_prices,
            "ask_vols": ask_vols,
            "bid_prices": bid_prices,
            "bid_vols": bid_vols,
            "close": close_vals.iloc[-1],
        })
    if skipped > 0:
        print(f"  Skipped {skipped} instruments (NaN level-1 at t=0 or missing close)")
    return instruments


train_instruments = extract_instruments(train_ids, Y_raw)
val_instruments = extract_instruments(val_ids, Y_raw)

print(f"Train instruments loaded: {len(train_instruments)}")
print(f"Val instruments loaded: {len(val_instruments)}")

## Gymnasium Environment

Each episode = one instrument's last K-second execution window (same window as TWAP-K and Oracle).  
State: 12 features (remaining vol/time, spread, depth, imbalance, momentum, etc.)  
Action: 6 discrete choices — multiples of the TWAP-K rate  

**Key design: agent only trades in the last K seconds.**  
This matches the Oracle's optimization window and TWAP-K's execution window.  
Executing earlier hurts BPS because prices drift from the close price.

**Reward (differential mark-to-market vs TWAP-K):**  
At each step, compare agent's cost-to-complete vs TWAP-K's cost-to-complete.  
Reward = change in advantage. Positive reward = agent is doing better than TWAP-K.

In [ ]:
# Quick check: instrument counts + TWAP-14 BPS on val instruments
print(f"Train: {len(train_instruments)}, Val: {len(val_instruments)}")
print(f"Target: ~475 train, ~119 val (matching mhf-final)")
print()

# Compute TWAP-14 BPS on val instruments to compare with submission (5.08)
twap_14_bps_list = []
for inst in val_instruments:
    positions = np.zeros(60)
    positions[-14:] = volume_to_fill / 14
    vol, avg = simulate_walk_the_book(
        positions, inst["ask_prices"], inst["ask_vols"],
        inst["bid_prices"], inst["bid_vols"]
    )
    if vol > 0 and not np.isnan(avg):
        ie = abs(avg - inst["close"]) / inst["close"] * 10000
        vp = min(100.0, volume_to_fill / vol)
        twap_14_bps_list.append(ie * vp)

print(f"TWAP-14 mean BPS on {len(twap_14_bps_list)} val instruments: {np.mean(twap_14_bps_list):.4f}")
print(f"Submission TWAP-14 BPS: 5.0781")

## Training

PPO (Proximal Policy Optimization) with a small MLP policy (2 hidden layers, 64 neurons each).
PPO uses trajectory rollouts rather than replay buffers, making it better suited for
environments with small per-step reward signals. RL-Exec (BTC-USD execution) used PPO
for this exact reason.

In [6]:
train_env = TradeExecutionEnv(train_instruments, volume_to_fill, K_SECONDS)

# Linear LR decay: 3e-4 → 0 over training
def linear_schedule(progress_remaining: float) -> float:
    return progress_remaining * 3e-4

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=linear_schedule,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs={"net_arch": [64, 64]},
    verbose=1,
    seed=SEED,
)

TOTAL_TIMESTEPS = 1_000_000  # ~59K episodes with K=17
print(f"Training for {TOTAL_TIMESTEPS} timesteps (~{TOTAL_TIMESTEPS // K_SECONDS} episodes)...")
print(f"LR schedule: 3e-4 → 0 (linear decay)")
model.learn(total_timesteps=TOTAL_TIMESTEPS)
print("Training complete.")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Training for 1000000 timesteps (~50000 episodes)...
LR schedule: 3e-4 → 0 (linear decay)
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 20       |
|    ep_rew_mean     | -0.411   |
| time/              |          |
|    fps             | 592      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 20          |
|    ep_rew_mean          | -0.471      |
| time/                   |             |
|    fps                  | 442         |
|    iterations           | 2           |
|    time_elapsed         | 9           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.012380403 |
|    clip_fraction        | 0.126       |
|    clip_range           | 0.2         |
|    entropy_loss        

## Evaluation

Run the trained agent on validation instruments and compare BPS against TWAP-K.  
Uses `simulate_walk_the_book()` for final BPS — same function as all other strategies.

In [7]:
def evaluate(model, instruments, volume_to_fill, K_seconds):
    """Evaluate RL agent vs multiple TWAP baselines on a set of instruments."""
    env = TradeExecutionEnv(instruments, volume_to_fill, K_seconds)
    rl_bps_list = []
    twap_k_bps_list = []
    twap_14_bps_list = []
    twap_60_bps_list = []
    rl_positions_all = []

    for i in range(len(instruments)):
        inst = instruments[i]
        ask_prices = inst["ask_prices"]
        ask_vols = inst["ask_vols"]
        bid_prices = inst["bid_prices"]
        bid_vols = inst["bid_vols"]
        close_price = inst["close"]

        # --- RL Agent ---
        obs, _ = env.reset(options={"instrument_idx": i})
        rl_positions = np.zeros(60)
        for step in range(K_seconds):
            action, _ = model.predict(obs, deterministic=True)

            position = env.action_multipliers[int(action)] * env.twap_rate
            max_remaining = volume_to_fill - env.volume_allocated
            position = min(position, max(0.0, max_remaining))
            if step == K_seconds - 1:
                position = max(0.0, max_remaining)

            abs_t = env.start_second + step
            rl_positions[abs_t] = position

            obs, _, done, _, _ = env.step(int(action))
            if done:
                break

        rl_positions_all.append(rl_positions)

        def compute_bps(positions):
            vol, avg = simulate_walk_the_book(
                positions, ask_prices, ask_vols, bid_prices, bid_vols
            )
            if vol > 0 and not np.isnan(avg):
                ie = abs(avg - close_price) / close_price * 10000
                vp = min(100.0, volume_to_fill / vol)
                return ie * vp
            return None

        # --- RL BPS ---
        bps = compute_bps(rl_positions)
        if bps is not None:
            rl_bps_list.append(bps)

        # --- TWAP-K ---
        twap_k_pos = np.zeros(60)
        twap_k_pos[-K_seconds:] = volume_to_fill / K_seconds
        bps = compute_bps(twap_k_pos)
        if bps is not None:
            twap_k_bps_list.append(bps)

        # --- TWAP-14 ---
        twap_14_pos = np.zeros(60)
        twap_14_pos[-14:] = volume_to_fill / 14
        bps = compute_bps(twap_14_pos)
        if bps is not None:
            twap_14_bps_list.append(bps)

        # --- TWAP-60 ---
        twap_60_pos = np.full(60, volume_to_fill / 60)
        bps = compute_bps(twap_60_pos)
        if bps is not None:
            twap_60_bps_list.append(bps)

    return (np.array(rl_bps_list), np.array(twap_k_bps_list),
            np.array(twap_14_bps_list), np.array(twap_60_bps_list),
            rl_positions_all)


rl_bps, twap_k_bps, twap_14_bps, twap_60_bps, rl_positions_all = evaluate(
    model, val_instruments, volume_to_fill, K_SECONDS
)

print(f"\n{'='*60}")
print(f"RESULTS: {data_asset} (volume={volume_to_fill})")
print(f"{'='*60}")
print(f"{'Strategy':<20} {'Mean BPS':>10} {'Median BPS':>12}")
print(f"{'-'*60}")
print(f"{'RL Agent':<20} {rl_bps.mean():>10.4f} {np.median(rl_bps):>12.4f}")
print(f"{'TWAP-' + str(K_SECONDS) + 's':<20} {twap_k_bps.mean():>10.4f} {np.median(twap_k_bps):>12.4f}")
print(f"{'TWAP-14s':<20} {twap_14_bps.mean():>10.4f} {np.median(twap_14_bps):>12.4f}")
print(f"{'TWAP-60s':<20} {twap_60_bps.mean():>10.4f} {np.median(twap_60_bps):>12.4f}")
print(f"{'-'*60}")

# Compare RL against each baseline
for name, baseline in [("TWAP-" + str(K_SECONDS), twap_k_bps),
                        ("TWAP-14", twap_14_bps),
                        ("TWAP-60", twap_60_bps)]:
    diff = baseline.mean() - rl_bps.mean()
    wins = (rl_bps < baseline).sum()
    pct = (rl_bps < baseline).mean() * 100
    label = "RL wins" if diff > 0 else "TWAP wins"
    print(f"RL vs {name:<8}: {diff:+.4f} bps ({label}), "
          f"RL wins {wins}/{len(rl_bps)} ({pct:.1f}%)")

print(f"{'='*60}")
print(f"Instruments evaluated: {len(rl_bps)}")


RESULTS: DOGEUSDT (volume=60349.0)
Strategy               Mean BPS   Median BPS
------------------------------------------------------------
RL Agent                 4.1818       3.8921
TWAP-20s                 4.2085       3.8979
TWAP-14s                 4.3539       3.7989
TWAP-60s                 5.8257       4.6451
------------------------------------------------------------
RL vs TWAP-20 : +0.0267 bps (RL wins), RL wins 36/66 (54.5%)
RL vs TWAP-14 : +0.1721 bps (RL wins), RL wins 35/66 (53.0%)
RL vs TWAP-60 : +1.6438 bps (RL wins), RL wins 37/66 (56.1%)
Instruments evaluated: 66


## Policy Analysis

Examine what the agent learned: average position profile across instruments,
compared to TWAP allocation.

In [8]:
# Average position profile
positions_array = np.array(rl_positions_all)  # [num_instruments, 60]
avg_positions = positions_array.mean(axis=0)

# TWAP-K profile for comparison
twap_profile = np.zeros(60)
twap_profile[-K_SECONDS:] = volume_to_fill / K_SECONDS

print("RL Agent — Average position profile (last K seconds):")
print(f"  Volume in last {K_SECONDS}s:   {avg_positions[-K_SECONDS:].sum():.4f} ({avg_positions[-K_SECONDS:].sum()/volume_to_fill*100:.1f}%)")
print(f"  Volume in last 5s:    {avg_positions[-5:].sum():.4f} ({avg_positions[-5:].sum()/volume_to_fill*100:.1f}%)")
print(f"  Volume at last sec:   {avg_positions[59]:.4f} ({avg_positions[59]/volume_to_fill*100:.1f}%)")
print()
print(f"Per-second breakdown (last {K_SECONDS} seconds):")
start = 60 - K_SECONDS
for s in range(start, 60):
    rl_avg = avg_positions[s]
    twap_avg = twap_profile[s]
    bar = '#' * int(rl_avg / volume_to_fill * 100)
    print(f"  t={s:2d}: RL={rl_avg:7.2f} TWAP={twap_avg:7.2f}  {bar}")
print()
print(f"Top 5 seconds by average position:")
top5 = np.argsort(avg_positions)[-5:][::-1]
for s in top5:
    print(f"  Second {s}: {avg_positions[s]:.4f} ({avg_positions[s]/volume_to_fill*100:.1f}%)")

RL Agent — Average position profile (last K seconds):
  Volume in last 20s:   60349.0000 (100.0%)
  Volume in last 5s:    16904.5778 (28.0%)
  Volume at last sec:   7509.3358 (12.4%)

Per-second breakdown (last 20 seconds):
  t=40: RL=3371.77 TWAP=3017.45  #####
  t=41: RL=3154.61 TWAP=3017.45  #####
  t=42: RL=3177.47 TWAP=3017.45  #####
  t=43: RL=2823.14 TWAP=3017.45  ####
  t=44: RL=2960.30 TWAP=3017.45  ####
  t=45: RL=3211.76 TWAP=3017.45  #####
  t=46: RL=2628.84 TWAP=3017.45  ####
  t=47: RL=3063.17 TWAP=3017.45  #####
  t=48: RL=2811.71 TWAP=3017.45  ####
  t=49: RL=2754.57 TWAP=3017.45  ####
  t=50: RL=2743.14 TWAP=3017.45  ####
  t=51: RL=2548.83 TWAP=3017.45  ####
  t=52: RL=2743.14 TWAP=3017.45  ####
  t=53: RL=2605.98 TWAP=3017.45  ####
  t=54: RL=2846.00 TWAP=3017.45  ####
  t=55: RL=2731.71 TWAP=3017.45  ####
  t=56: RL=2377.38 TWAP=3017.45  ###
  t=57: RL=2343.10 TWAP=3017.45  ###
  t=58: RL=1943.05 TWAP=3017.45  ###
  t=59: RL=7509.34 TWAP=3017.45  ############

Top 5

## Generate Test Positions

Run the agent on test data and save positions in the submission format.

In [9]:
# Generate positions for validation instruments (demo — replace with test data when available)
print("Generating positions for validation instruments...")

times = Y_raw["time_in_hour"].sort_values().unique()

test_positions_df = pd.DataFrame()
env = TradeExecutionEnv(val_instruments, volume_to_fill, K_SECONDS)

for i, inst in enumerate(val_instruments):
    obs, _ = env.reset(options={"instrument_idx": i})
    positions = np.zeros(60)
    for step in range(K_SECONDS):
        action, _ = model.predict(obs, deterministic=True)
        pos = env.action_multipliers[int(action)] * env.twap_rate
        max_rem = volume_to_fill - env.volume_allocated
        pos = min(pos, max(0.0, max_rem))
        if step == K_SECONDS - 1:
            pos = max(0.0, max_rem)

        abs_t = env.start_second + step
        positions[abs_t] = pos
        obs, _, done, _, _ = env.step(int(action))
        if done:
            break

    df = pd.DataFrame({
        'anonymized_id': inst['id'],
        'time_in_hour': times,
        'position': positions,
    })
    test_positions_df = pd.concat([test_positions_df, df], ignore_index=True)

base_path = Path(f"positions/{data_asset}")
base_path.mkdir(parents=True, exist_ok=True)
target = base_path / "rl_agent.parquet"
test_positions_df.to_parquet(target, index=False, engine='pyarrow')
print(f"Saved {len(test_positions_df)} rows to {target}")

Generating positions for validation instruments...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved 3960 rows to positions/DOGEUSDT/rl_agent.parquet
